In [1]:
import os 
from dotenv import load_dotenv
from openai import OpenAI
from scipy.spatial import distance
import numpy as np

In [3]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

In [5]:
articles= [
    {
"headline":"AI is transforming healthcare",
"topic":"Technology",
"keywords": ["AI","machine learning","healthcare"]
    },
    {
"headline":"New GPU architecture released",
"topic":"Hardware",
"keywords": ["GPU","computing","graphics"]
    },
    {
"headline":"Climate change impacts agriculture",
"topic":"Environment",
"keywords": ["climate","agriculture","environment"]
    }
]

In [6]:
def create_article_text(article):
		return f"""
    Headline:{article['headline']}
    Topic:{article['topic']}
    Keywords:{', '.join(article['keywords'])}
    """

articles_texts = [create_article_text(article) for article in articles ]

In [11]:
print(articles_texts)

['\n    Headline:AI is transforming healthcare\n    Topic:Technology\n    Keywords:AI, machine learning, healthcare\n    ', '\n    Headline:New GPU architecture released\n    Topic:Hardware\n    Keywords:GPU, computing, graphics\n    ', '\n    Headline:Climate change impacts agriculture\n    Topic:Environment\n    Keywords:climate, agriculture, environment\n    ']


In [12]:
def create_embeddings(texts):
	response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
    )
	return [item.embedding for item in response.data]

article_embeddings=create_embeddings(articles_texts)

#print(article_embeddings)

     

In [13]:
def cosine_distance(v1, v2):
    v1 = np.array(v1)
    v2 = np.array(v2)
    return distance.cosine(v1, v2)

In [14]:
def find_n_closest(query_vector, embeddings, n=3):

    distances= []

    for i, embedding in enumerate(embeddings):
        distance=cosine_distance(query_vector,embedding)

        distances.append({
       "index":i,
        "distance":distance
        })

    distances=sorted(distances, key=lambda x:x["distance"]) 

    return distances[:n]

In [15]:
query_embedding=create_embeddings(["AI"])[0]
print(query_embedding)


[-0.0081634521484375, -0.0246124267578125, 0.0029850006103515625, 0.0251617431640625, 0.006565093994140625, -0.028228759765625, -0.005023956298828125, 0.020904541015625, -0.036895751953125, 0.01279449462890625, -0.0030364990234375, -0.020111083984375, 0.0002522468566894531, -0.03271484375, 0.0064544677734375, -0.0252685546875, -0.031097412109375, -0.054412841796875, 0.03277587890625, -0.0184173583984375, 0.01666259765625, 0.04833984375, -0.024871826171875, 0.01438140869140625, 0.0293426513671875, 0.004047393798828125, 0.00928497314453125, 0.01337432861328125, 0.0025310516357421875, -0.0225372314453125, 0.0321044921875, -0.0280303955078125, 0.005359649658203125, -0.038177490234375, -0.0167236328125, 0.01434326171875, -0.038604736328125, -0.01038360595703125, -0.0105438232421875, -0.0191650390625, 0.0321044921875, 0.014556884765625, -0.021514892578125, 0.0160675048828125, -0.01186370849609375, 0.0013990402221679688, -0.004833221435546875, -0.033660888671875, -0.02581787109375, 0.04565429

In [16]:
hits=find_n_closest(query_embedding,article_embeddings,n=3)

print(hits)

[{'index': 0, 'distance': np.float64(0.6439031903735453)}, {'index': 1, 'distance': np.float64(0.7846324459599807)}, {'index': 2, 'distance': np.float64(0.8272761614076926)}]


In [17]:
for hit in hits:
    print(articles[hit["index"]]["headline"])

AI is transforming healthcare
New GPU architecture released
Climate change impacts agriculture
